# 03 — Análisis de Componentes Principales (PCA)

**Materia:** Inteligencia Artificial

**Proyecto:** Clasificación del uso de gas en garrafa en los hogares de Bolivia.

**Fuente de datos:** Encuesta de Hogares 2023 — INE Bolivia.

**Responsable:** Integrante 2 — Preprocesamiento y PCA.

## Objetivo

Aplicar el Análisis de Componentes Principales (PCA) para reducir la dimensionalidad del conjunto de datos preprocesado, evaluando diferentes cantidades de componentes principales y analizando la proporción de varianza explicada.

Se realizarán cinco experimentos con 12, 10, 9, 5 y 3 componentes principales, con la finalidad de proporcionar diferentes representaciones de los datos para su posterior utilización en los algoritmos de clasificación y agrupamiento.

In [ ]:
# 1. Importación de librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import sys
import importlib

from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

print("Librerías importadas correctamente")

In [ ]:
# 2. Conexión con Google Drive

from google.colab import drive

drive.mount('/content/drive')

ruta = '/content/drive/MyDrive/PROYECTO_DE_IA/'

In [ ]:
# 3. Importación del módulo de preprocesamiento

carpeta_src = Path(ruta) / 'src'

if str(carpeta_src) not in sys.path:
    sys.path.insert(0, str(carpeta_src))

import preprocessing

importlib.reload(preprocessing)

print("Módulo preprocessing importado correctamente")

In [ ]:
# 4. Carga del conjunto de datos limpio

ruta_csv = ruta + 'data/processed/datos_limpios.csv'

X, Y = preprocessing.cargar_datos(ruta_csv)

print("Dimensiones de X:", X.shape)

print("Dimensiones de Y:", Y.shape)

print("\nDistribución de la variable objetivo:")

display(Y.value_counts().sort_index())

display(X.head())

## Fundamento matemático del Análisis de Componentes Principales

El Análisis de Componentes Principales (PCA) es una técnica de aprendizaje no supervisado que permite reducir la dimensionalidad de un conjunto de datos, transformando las variables originales en nuevas variables denominadas componentes principales.

Estos componentes son combinaciones lineales de las características de entrada y se encuentran ordenados según la cantidad de varianza que explican.

En este proyecto, después de aplicar One-Hot Encoding y StandardScaler, se obtuvieron 62 características numéricas. Mediante PCA se busca reducir esta dimensionalidad, conservando la mayor proporción posible de varianza.

Para comprender su funcionamiento se analizarán la matriz de covarianza, los valores propios, los vectores propios y la varianza explicada.

In [ ]:
# 5. División de los datos

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    stratify=Y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
# 6. Aplicación del preprocesamiento

preprocesador = preprocessing.crear_preprocesador()

X_train_procesado = preprocesador.fit_transform(
    X_train
)

X_test_procesado = preprocesador.transform(
    X_test
)

nombres_columnas = preprocesador.get_feature_names_out()

print("Entrenamiento procesado:",
      X_train_procesado.shape)

print("Prueba procesada:",
      X_test_procesado.shape)

In [ ]:
# 7. Centrado de los datos

media = np.mean(
    X_train_procesado,
    axis=0
)

X_centrado = X_train_procesado - media

print("Dimensiones de la matriz centrada:",
      X_centrado.shape)

print("\nMedia de las primeras 10 características:")

print(
    np.round(
        X_centrado.mean(axis=0)[:10],
        6
    )
)

In [ ]:
# 8. Cálculo de la matriz de covarianza

n = X_centrado.shape[0]

matriz_covarianza = (
    X_centrado.T @ X_centrado
) / (n - 1)

print("Dimensiones de la matriz de covarianza:",
      matriz_covarianza.shape)

covarianza_df = pd.DataFrame(
    matriz_covarianza,
    index=nombres_columnas,
    columns=nombres_columnas
)

display(covarianza_df.iloc[:8, :8].round(4))

In [ ]:
# 9. Visualización de la matriz de covarianza

plt.figure(figsize=(12, 10))

plt.imshow(
    matriz_covarianza,
    cmap='coolwarm',
    aspect='auto'
)

plt.colorbar(label='Covarianza')

plt.title(
    'Matriz de covarianza de las 62 características'
)

plt.xlabel('Características')

plt.ylabel('Características')

plt.tight_layout()

plt.show()

In [ ]:
# 10. Cálculo de valores y vectores propios

valores_propios, vectores_propios = np.linalg.eigh(
    matriz_covarianza
)

# Ordenar de mayor a menor

orden = np.argsort(valores_propios)[::-1]

valores_propios = valores_propios[orden]

vectores_propios = vectores_propios[:, orden]

print("Cantidad de valores propios:",
      len(valores_propios))

print("Dimensiones de los vectores propios:",
      vectores_propios.shape)

print("\nPrimeros 12 valores propios:")

display(
    pd.DataFrame({
        'Componente': [
            f'PC{i+1}' for i in range(12)
        ],
        'Valor propio': valores_propios[:12]
    })
)

In [ ]:
# 11. Cálculo de la varianza explicada

varianza_explicada = (
    valores_propios / valores_propios.sum()
) * 100

varianza_acumulada = np.cumsum(
    varianza_explicada
)

tabla_varianza = pd.DataFrame({
    'Componente': [
        f'PC{i+1}'
        for i in range(len(valores_propios))
    ],

    'Valor propio': valores_propios,

    'Varianza explicada (%)': varianza_explicada,

    'Varianza acumulada (%)': varianza_acumulada
})

display(tabla_varianza.head(12).round(4))

print("\nVarianza total:",
      round(varianza_explicada.sum(), 6), "%")

## Validación matemática y experimentación con PCA

A partir de la matriz de covarianza se calcularon los valores y vectores propios de las 62 características preprocesadas.

Los resultados obtenidos muestran que los primeros 12 componentes principales conservan aproximadamente el 79,36 % de la varianza total.

Para verificar la implementación matemática realizada mediante NumPy, se compararán los valores propios y las proporciones de varianza explicada con los resultados obtenidos mediante la biblioteca Scikit-learn.

Posteriormente, se realizarán cinco experimentos utilizando 12, 10, 9, 5 y 3 componentes principales, manteniendo la misma partición de entrenamiento y prueba para facilitar la comparación.

In [ ]:
# 12. Validación matemática con Scikit-learn

pca_completo = PCA(svd_solver='full')

pca_completo.fit(X_train_procesado)

coincidencia_valores = np.allclose(
    valores_propios,
    pca_completo.explained_variance_
)

coincidencia_varianza = np.allclose(
    varianza_explicada,
    pca_completo.explained_variance_ratio_ * 100
)

print("Coincidencia de valores propios:",
      coincidencia_valores)

print("Coincidencia de varianza explicada:",
      coincidencia_varianza)

assert coincidencia_valores
assert coincidencia_varianza

print("\nValidación matemática exitosa")

In [ ]:
# 13. Proyección matemática con 12 componentes

k = 12

W = vectores_propios[:, :k]

# Proyección del entrenamiento
X_train_pca_manual = X_centrado @ W

# Centrar prueba usando la media de entrenamiento
X_test_centrado = X_test_procesado - media

# Proyección de prueba
X_test_pca_manual = X_test_centrado @ W

print("Dimensiones de W:", W.shape)

print("Entrenamiento PCA:",
      X_train_pca_manual.shape)

print("Prueba PCA:",
      X_test_pca_manual.shape)

display(
    pd.DataFrame(
        X_train_pca_manual,
        columns=[f'PC{i+1}' for i in range(k)]
    ).head()
)

In [ ]:
# 14. Experimentos con diferentes componentes PCA

componentes_experimentos = [12, 10, 9, 5, 3]

resultados = []

modelos_pca = {}

datos_pca = {}

for k in componentes_experimentos:

    # Crear modelo PCA
    pca = PCA(
        n_components=k,
        svd_solver='full'
    )

    # Ajustar solamente con entrenamiento
    X_train_pca = pca.fit_transform(
        X_train_procesado
    )

    # Transformar los datos de prueba
    X_test_pca = pca.transform(
        X_test_procesado
    )

    # Calcular varianza conservada
    varianza = (
        pca.explained_variance_ratio_.sum() * 100
    )

    # Calcular reducción dimensional
    reduccion = (
        1 - k / X_train_procesado.shape[1]
    ) * 100

    # Guardar resultados
    resultados.append({
        'Componentes': k,
        'Varianza conservada (%)': varianza,
        'Varianza no conservada (%)': 100 - varianza,
        'Reducción dimensional (%)': reduccion
    })

    # Guardar modelos y datos transformados
    modelos_pca[k] = pca

    datos_pca[k] = {
        'X_train': X_train_pca,
        'X_test': X_test_pca
    }

    print(f"\nPCA con {k} componentes")
    print("Entrenamiento:", X_train_pca.shape)
    print("Prueba:", X_test_pca.shape)
    print(f"Varianza conservada: {varianza:.4f}%")

In [ ]:
# 15. Tabla comparativa de resultados PCA

resultados_pca = pd.DataFrame(resultados)

display(resultados_pca.round(4))

In [ ]:
# 16. Validación de los experimentos PCA

for k in componentes_experimentos:

    X_entrenamiento = datos_pca[k]['X_train']
    X_prueba = datos_pca[k]['X_test']

    assert X_entrenamiento.shape == (10252, k)

    assert X_prueba.shape == (2563, k)

    assert np.isfinite(X_entrenamiento).all()

    assert np.isfinite(X_prueba).all()

    varianza_sklearn = (
        modelos_pca[k]
        .explained_variance_ratio_.sum() * 100
    )

    varianza_manual = varianza_acumulada[k - 1]

    assert np.isclose(
        varianza_sklearn,
        varianza_manual
    )

    print(f"PCA con {k} componentes: Validación exitosa")

print("\nTodos los experimentos fueron validados")

## Visualización e interpretación de los resultados de PCA

Una vez validados los experimentos de reducción de dimensionalidad, se generarán representaciones gráficas que permitan analizar la varianza explicada individual, la varianza acumulada y la comparación entre las diferentes cantidades de componentes principales.

También se realizará una representación bidimensional de los hogares mediante los dos primeros componentes principales, diferenciando aquellos que utilizan gas en garrafa como combustible principal de aquellos que no lo utilizan.

Los gráficos permitirán interpretar los resultados y servirán como material de apoyo para el artículo científico y la defensa del proyecto.

In [ ]:
# 17. Preparación de las carpetas de resultados

ruta_figuras = Path(ruta) / 'results' / 'figures'

ruta_tablas = Path(ruta) / 'results' / 'tables'

ruta_figuras.mkdir(parents=True, exist_ok=True)

ruta_tablas.mkdir(parents=True, exist_ok=True)

print("Carpeta de figuras:", ruta_figuras)

print("Carpeta de tablas:", ruta_tablas)

print("\nCarpetas preparadas correctamente")

In [ ]:
# 18. Gráfico de varianza explicada individual

plt.figure(figsize=(12, 6))

plt.bar(
    range(1, len(varianza_explicada) + 1),
    varianza_explicada,
    color='steelblue',
    edgecolor='black',
    linewidth=0.4
)

plt.title('Varianza explicada individual por componente')

plt.xlabel('Componente principal')

plt.ylabel('Varianza explicada (%)')

plt.grid(axis='y', alpha=0.3)

plt.tight_layout()

plt.savefig(
    ruta_figuras / 'varianza_individual.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# 19. Gráfico de varianza acumulada

plt.figure(figsize=(12, 6))

plt.plot(
    range(1, len(varianza_acumulada) + 1),
    varianza_acumulada,
    marker='o',
    markersize=3,
    color='steelblue',
    label='Varianza acumulada'
)

plt.axhline(
    y=80,
    color='red',
    linestyle='--',
    label='Referencia del 80%'
)

for k in componentes_experimentos:

    plt.scatter(
        k,
        varianza_acumulada[k - 1],
        color='darkorange',
        s=70,
        zorder=3
    )

plt.title('Varianza explicada acumulada de PCA')

plt.xlabel('Número de componentes principales')

plt.ylabel('Varianza acumulada (%)')

plt.ylim(0, 105)

plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()

plt.savefig(
    ruta_figuras / 'varianza_acumulada.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# 20. Comparación de los experimentos PCA

plt.figure(figsize=(10, 6))

etiquetas = resultados_pca['Componentes'].astype(str)

barras = plt.bar(
    etiquetas,
    resultados_pca['Varianza conservada (%)'],
    color='steelblue',
    edgecolor='black'
)

plt.bar_label(
    barras,
    fmt='%.2f%%',
    padding=4
)

plt.title('Comparación de varianza conservada por PCA')

plt.xlabel('Número de componentes principales')

plt.ylabel('Varianza conservada (%)')

plt.ylim(0, 100)

plt.grid(axis='y', alpha=0.3)

plt.tight_layout()

plt.savefig(
    ruta_figuras / 'comparacion_pca.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# 21. Visualización bidimensional de los hogares

X_visualizacion = datos_pca[12]['X_train']

Y_visualizacion = Y_train.to_numpy()

plt.figure(figsize=(11, 7))

for clase, etiqueta, color in [
    (0, 'No utiliza garrafa como combustible principal', 'steelblue'),
    (1, 'Utiliza garrafa como combustible principal', 'darkorange')
]:

    mascara = Y_visualizacion == clase

    plt.scatter(
        X_visualizacion[mascara, 0],
        X_visualizacion[mascara, 1],
        label=etiqueta,
        color=color,
        alpha=0.25,
        s=10
    )

plt.title('Representación de hogares mediante PCA')

plt.xlabel(
    f'PC1 ({varianza_explicada[0]:.2f}%)'
)

plt.ylabel(
    f'PC2 ({varianza_explicada[1]:.2f}%)'
)

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    ruta_figuras / 'dispersion_pca.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# 22. Exportación del gráfico de covarianza

plt.figure(figsize=(12, 10))

limite = np.abs(matriz_covarianza).max()

plt.imshow(
    matriz_covarianza,
    cmap='coolwarm',
    vmin=-limite,
    vmax=limite,
    aspect='auto'
)

plt.colorbar(label='Covarianza')

plt.title('Matriz de covarianza de las 62 características')

plt.xlabel('Características')

plt.ylabel('Características')

plt.tight_layout()

plt.savefig(
    ruta_figuras / 'matriz_covarianza.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# 23. Verificación de los gráficos generados

archivos_esperados = [
    'varianza_individual.png',
    'varianza_acumulada.png',
    'comparacion_pca.png',
    'dispersion_pca.png',
    'matriz_covarianza.png'
]

for nombre in archivos_esperados:

    archivo = ruta_figuras / nombre

    assert archivo.is_file()

    assert archivo.stat().st_size > 0

    print(f"{nombre}: Guardado correctamente")

print("\nTodos los gráficos fueron generados")

## Exportación y validación de los resultados

Los resultados obtenidos mediante los experimentos de PCA serán exportados en formato CSV para facilitar su análisis, incorporación al artículo científico y utilización por los demás integrantes del equipo.

Se almacenarán los valores propios, la varianza explicada individual y acumulada, los coeficientes de los componentes principales y la comparación de los cinco experimentos de reducción de dimensionalidad.

Finalmente, se verificará la integración del módulo de preprocesamiento y PCA con un algoritmo de clasificación, garantizando que las transformaciones se ajusten exclusivamente con los datos de entrenamiento.

In [ ]:
# 24. Exportación de las tablas de PCA

tabla_varianza.to_csv(
    ruta_tablas / 'varianza_componentes.csv',
    index=False
)

resultados_pca.to_csv(
    ruta_tablas / 'resultados_pca.csv',
    index=False
)

print("Tablas guardadas correctamente")

print("\nTabla comparativa de PCA:")

display(resultados_pca.round(4))

In [ ]:
# 25. Coeficientes de los componentes principales

coeficientes_pca = pd.DataFrame(
    modelos_pca[12].components_.T,
    index=nombres_columnas,
    columns=[f'PC{i+1}' for i in range(12)]
)

coeficientes_pca.index.name = 'Caracteristica'

coeficientes_pca.to_csv(
    ruta_tablas / 'coeficientes_pca.csv'
)

print("Dimensiones:", coeficientes_pca.shape)

display(coeficientes_pca.head(10).round(4))

In [ ]:
# 26. Interpretación de los componentes principales

for componente in ['PC1', 'PC2', 'PC3']:

    principales = (
        coeficientes_pca[componente]
        .abs()
        .nlargest(5)
        .index
    )

    tabla = pd.DataFrame({
        'Coeficiente':
            coeficientes_pca.loc[principales, componente],

        'Magnitud absoluta':
            coeficientes_pca.loc[principales, componente].abs()
    })

    print(f"\nCaracterísticas principales de {componente}")

    display(tabla.round(4))

In [ ]:
# 27. Verificación de las tablas exportadas

tablas_esperadas = [
    'varianza_componentes.csv',
    'resultados_pca.csv',
    'coeficientes_pca.csv'
]

for nombre in tablas_esperadas:

    archivo = ruta_tablas / nombre

    assert archivo.is_file()

    assert archivo.stat().st_size > 0

    tabla = pd.read_csv(archivo)

    print(f"{nombre}: Guardado correctamente")
    print("Dimensiones:", tabla.shape)
    print()

print("Todas las tablas fueron exportadas")

## Prueba de integración con los algoritmos de clasificación

Se realizará una prueba funcional del módulo preprocessing.py mediante un árbol de decisión.

El propósito de esta prueba es verificar que el preprocesamiento, PCA y el algoritmo de clasificación puedan integrarse correctamente dentro de un pipeline.

Esta prueba no constituye la evaluación definitiva de los modelos de clasificación, ya que dicha evaluación será desarrollada posteriormente mediante múltiples particiones y métricas de rendimiento por el integrante responsable.

In [ ]:
# 28. Prueba de integración con clasificación

from sklearn.tree import DecisionTreeClassifier

pipeline_prueba = preprocessing.crear_pipeline(
    n_componentes=12,
    modelo=DecisionTreeClassifier(
        random_state=42
    )
)

# Entrenamiento del pipeline
pipeline_prueba.fit(X_train, Y_train)

# Generar predicciones
predicciones = pipeline_prueba.predict(X_test)

print("Pipeline entrenado correctamente")

print("\nCantidad de predicciones:",
      len(predicciones))

print("Clases predichas:",
      np.unique(predicciones))

assert len(predicciones) == len(Y_test)

assert np.isin(predicciones, [0, 1]).all()

print("\nPrueba de integración exitosa")

In [ ]:
# 29. Validación de todas las configuraciones

configuraciones = [None, 12, 10, 9, 5, 3]

for k in configuraciones:

    pipeline = preprocessing.crear_pipeline(
        n_componentes=k,
        modelo=DecisionTreeClassifier(
            random_state=42
        )
    )

    pipeline.fit(X_train, Y_train)

    pred = pipeline.predict(X_test)

    assert len(pred) == len(Y_test)

    assert np.isin(pred, [0, 1]).all()

    nombre = 'Sin PCA' if k is None else f'PCA con {k} componentes'

    print(f"{nombre}: Funcionamiento correcto")

print("\nTodas las configuraciones funcionan correctamente")

## Conclusiones del preprocesamiento y PCA

Durante el preprocesamiento de la Encuesta de Hogares 2023 del INE Bolivia se analizaron 12.815 registros y 37 columnas originales.

Se seleccionaron 12 variables predictoras relacionadas con las características de los hogares y se construyó una variable objetivo binaria para identificar el uso de gas en garrafa como combustible principal para cocinar.

Se identificó una inconsistencia en las etiquetas de la variable s06a_05, correspondiente al material del techo de la vivienda. La corrección se realizó mediante la extracción de los códigos numéricos y la comparación de sus frecuencias ponderadas con las publicadas en el diccionario oficial del INE.

Mediante One-Hot Encoding y StandardScaler, las 12 variables predictoras se transformaron en 62 características numéricas, sin presentar valores faltantes en los conjuntos de entrenamiento y prueba.

Posteriormente, se implementó matemáticamente el Análisis de Componentes Principales mediante el cálculo de la matriz de covarianza, los valores propios y los vectores propios. Los resultados fueron validados utilizando la biblioteca Scikit-learn.

Los experimentos permitieron obtener los siguientes porcentajes de varianza conservada:

- 12 componentes: 79,36 %.
- 10 componentes: 74,82 %.
- 9 componentes: 72,03 %.
- 5 componentes: 57,25 %.
- 3 componentes: 45,13 %.

La configuración de 12 componentes permitió reducir la dimensionalidad en un 80,65 %, conservando aproximadamente el 79,36 % de la varianza total.

Sin embargo, una mayor proporción de varianza conservada no garantiza un mejor rendimiento predictivo. Por este motivo, las cinco configuraciones deberán evaluarse posteriormente mediante los algoritmos de clasificación.

Finalmente, se desarrolló y validó el módulo preprocessing.py, permitiendo reutilizar las transformaciones dentro de un pipeline y evitando fugas de información durante los experimentos.

## Limitaciones

Los resultados de PCA corresponden a una partición inicial del 80 % para entrenamiento y 20 % para prueba, utilizando una semilla aleatoria de 42.

Se estandarizaron únicamente las variables cuantitativas, manteniendo las variables categóricas en formato binario.

La corrección de s06a_05 se encuentra respaldada por la coincidencia de las frecuencias ponderadas oficiales, aunque no se realizó una comparación individual con los registros originales sin etiquetas.

Los experimentos se realizaron sin ponderación muestral en el ajuste de PCA, por lo que sus resultados describen la muestra analizada y no constituyen estimaciones representativas de toda la población boliviana.

In [ ]:
# 30. Documentación del integrante 2

from pathlib import Path

documentacion = """
# Integrante 2 - Preprocesamiento y PCA

## Proyecto

Clasificación del uso de gas en garrafa
en los hogares de Bolivia.

Fuente: Encuesta de Hogares 2023 - INE Bolivia.

## Datos

- Registros: 12815
- Columnas originales: 37
- Variables predictoras: 12
- Variable objetivo: usa_garrafa
- Hogares que usan garrafa: 6605
- Hogares que no usan garrafa como combustible principal: 6210

## Preprocesamiento

- Revisión de valores faltantes.
- Verificación de registros duplicados.
- Corrección de etiquetas de s06a_05.
- Codificación One-Hot de variables categóricas.
- Estandarización de variables numéricas.
- Generación de 62 características.

## Resultados de PCA

| Componentes | Varianza conservada |
|-------------|---------------------|
| 12          | 79.36%              |
| 10          | 74.82%              |
| 9           | 72.03%              |
| 5           | 57.25%              |
| 3           | 45.13%              |

## Orden de ejecución

1. notebooks/02_preprocesamiento.ipynb
2. notebooks/03_pca.ipynb

## Archivos principales

- src/preprocessing.py
- data/processed/datos_limpios.csv
- results/figures/
- results/tables/

## Uso del módulo

Desde la carpeta raíz del proyecto:

    from src.preprocessing import cargar_datos
    from src.preprocessing import crear_pipeline
    from sklearn.tree import DecisionTreeClassifier

    X, Y = cargar_datos(
        'data/processed/datos_limpios.csv'
    )

    pipeline = crear_pipeline(
        n_componentes=12,
        modelo=DecisionTreeClassifier(random_state=42)
    )

El pipeline debe ajustarse únicamente con los
datos de entrenamiento de cada partición.

Para trabajar sin PCA, utilizar:

    n_componentes=None

Para los experimentos con PCA, utilizar:

12, 10, 9, 5 o 3 componentes.

## Consideraciones metodológicas

- Partición inicial 80/20 con semilla 42.
- División estratificada de los datos.
- Estandarización únicamente de variables numéricas.
- Variables categóricas codificadas mediante One-Hot.
- PCA ajustado exclusivamente con entrenamiento.
- Experimentos realizados sin ponderación muestral.
- Corrección de etiquetas de s06a_05 respaldada
  por la comparación de frecuencias ponderadas oficiales.
- No se eliminaron hogares con características iguales
  porque sus folios originales eran únicos.

## Limitaciones

Una mayor varianza conservada no garantiza
un mejor rendimiento en clasificación.

Los datos corresponden a una encuesta transversal
del año 2023.

Los datos originales deben obtenerse del INE.
No redistribuir los CSV sin verificar sus permisos.
"""

archivo_readme = Path(ruta) / 'README_integrante2.md'

archivo_readme.write_text(
    documentacion,
    encoding='utf-8'
)

print("Documentación creada correctamente")
print("Ubicación:", archivo_readme)

In [ ]:
# 31. Registro de dependencias

from importlib.metadata import version
from pathlib import Path

librerias = [
    'numpy',
    'pandas',
    'matplotlib',
    'scikit-learn'
]

dependencias = [
    f"{libreria}=={version(libreria)}"
    for libreria in librerias
]

archivo_dependencias = (
    Path(ruta) / 'requirements_integrante2.txt'
)

archivo_dependencias.write_text(
    '\n'.join(dependencias) + '\n',
    encoding='utf-8'
)

print("Dependencias registradas:\n")

for dependencia in dependencias:
    print(dependencia)

print("\nArchivo guardado correctamente")
print("Ubicación:", archivo_dependencias)

In [ ]:
# 32. Verificación final del integrante 2

from pathlib import Path
import pandas as pd

archivos = [
    'data/raw/EH2023_Vivienda.csv',
    'data/processed/datos_limpios.csv',

    'notebooks/02_preprocesamiento.ipynb',
    'notebooks/03_pca.ipynb',

    'src/preprocessing.py',

    'results/figures/matriz_covarianza.png',
    'results/figures/varianza_individual.png',
    'results/figures/varianza_acumulada.png',
    'results/figures/comparacion_pca.png',
    'results/figures/dispersion_pca.png',

    'results/tables/varianza_componentes.csv',
    'results/tables/resultados_pca.csv',
    'results/tables/coeficientes_pca.csv',

    'README_integrante2.md',
    'requirements_integrante2.txt'
]

verificacion_final = []

for nombre in archivos:

    archivo = Path(ruta) / nombre

    existe = (
        archivo.is_file()
        and archivo.stat().st_size > 0
    )

    verificacion_final.append({
        'Archivo': nombre,
        'Estado': 'Correcto' if existe else 'Revisar'
    })

tabla_final = pd.DataFrame(verificacion_final)

display(tabla_final)

correctos = (
    tabla_final['Estado'] == 'Correcto'
).sum()

print(f"\nArchivos correctos: {correctos}/{len(archivos)}")

if correctos == len(archivos):
    print("Todos los entregables están presentes")
else:
    print("Existen archivos pendientes de revisión")